In [11]:
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder












In [12]:
# Load MNIST dataset
mnist = fetch_openml('mnist_784', version=1)

X = mnist.data.values / 255.0
y = mnist.target.astype(int).values.reshape(-1,1)

# One hot encoding
encoder = OneHotEncoder(sparse_output=False)
y = encoder.fit_transform(y)

# Train test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [13]:
# Activation functions
def relu(x):
    return np.maximum(0, x)

def relu_derivative(x):
    return (x > 0).astype(float)

def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

# Network structure
input_size = 784
hidden1 = 128
hidden2 = 64
output_size = 10


In [14]:
# He initialization (better for ReLU)
W1 = np.random.randn(input_size, hidden1) * np.sqrt(2/input_size)
b1 = np.zeros((1, hidden1))

W2 = np.random.randn(hidden1, hidden2) * np.sqrt(2/hidden1)
b2 = np.zeros((1, hidden2))

W3 = np.random.randn(hidden2, output_size) * np.sqrt(2/hidden2)
b3 = np.zeros((1, output_size))

learning_rate = 0.01
epochs = 20

m = X_train.shape[0]


In [15]:
# Training loop
for epoch in range(epochs):

    # Forward propagation
    Z1 = np.dot(X_train, W1) + b1
    A1 = relu(Z1)

    Z2 = np.dot(A1, W2) + b2
    A2 = relu(Z2)

    Z3 = np.dot(A2, W3) + b3
    A3 = softmax(Z3)

    # Cross entropy loss
    loss = -np.mean(np.sum(y_train * np.log(A3 + 1e-8), axis=1))

    # Backpropagation
    dZ3 = A3 - y_train
    dW3 = np.dot(A2.T, dZ3) / m
    db3 = np.sum(dZ3, axis=0, keepdims=True) / m

    dA2 = np.dot(dZ3, W3.T)
    dZ2 = dA2 * relu_derivative(Z2)
    dW2 = np.dot(A1.T, dZ2) / m
    db2 = np.sum(dZ2, axis=0, keepdims=True) / m

    dA1 = np.dot(dZ2, W2.T)
    dZ1 = dA1 * relu_derivative(Z1)
    dW1 = np.dot(X_train.T, dZ1) / m
    db1 = np.sum(dZ1, axis=0, keepdims=True) / m

    # Update weights
    W3 -= learning_rate * dW3
    b3 -= learning_rate * db3

    W2 -= learning_rate * dW2
    b2 -= learning_rate * db2

    W1 -= learning_rate * dW1
    b1 -= learning_rate * db1

    print(f"Epoch {epoch+1}, Loss: {loss}")


Epoch 1, Loss: 2.381118441094568
Epoch 2, Loss: 2.369787316863313
Epoch 3, Loss: 2.3588720983075637
Epoch 4, Loss: 2.3483402747088538
Epoch 5, Loss: 2.338180972097603
Epoch 6, Loss: 2.328344214770752
Epoch 7, Loss: 2.31881723078736
Epoch 8, Loss: 2.3095694473484714
Epoch 9, Loss: 2.3005862255390572
Epoch 10, Loss: 2.2918296283751403
Epoch 11, Loss: 2.2832900775030542
Epoch 12, Loss: 2.2749595258743627
Epoch 13, Loss: 2.2668149946294514
Epoch 14, Loss: 2.2588600417923983
Epoch 15, Loss: 2.251070087330401
Epoch 16, Loss: 2.2434268801556025
Epoch 17, Loss: 2.235913497110514
Epoch 18, Loss: 2.228507801560715
Epoch 19, Loss: 2.22120300851303
Epoch 20, Loss: 2.2139987714324953


In [16]:
# Testing

Z1 = np.dot(X_test, W1) + b1
A1 = relu(Z1)

Z2 = np.dot(A1, W2) + b2
A2 = relu(Z2)

Z3 = np.dot(A2, W3) + b3
A3 = softmax(Z3)

predictions = np.argmax(A3, axis=1)
true_labels = np.argmax(y_test, axis=1)

accuracy = np.mean(predictions == true_labels)

print("Test Accuracy:", accuracy)


Test Accuracy: 0.23235714285714285
